In [1]:
# Cell 1 - set up
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="cuda" 
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [2]:
# Cell 2 - zero shot prompt

def zero_shot_prompt(question):
    prompt = f"""Solve the following math problems. Show step by step. Always end your answer with #### <number> where <number> is the final answer.

Q: {question}
A:"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # ← must not be commented out

    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.3,
        top_p=0.95
    )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = output[0][input_length:]  # ← only decode new tokens
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

In [3]:
# Cell 3 - test zero shot prompt
generated_answer = zero_shot_prompt("There are 12 crates that each contain 150 oranges. \
There are 16 boxes that each hold 30 nectarines. \
How many pieces of fruit are in total?")
print(generated_answer)

To find the total number of pieces of fruit, we need to calculate the number of oranges and the number of nectarines separately, then add them together.

### Step 1: Calculate the total number of oranges

- Number of crates: \( 12 \)
- Oranges per crate: \( 150 \)

\[
\text{Total number of oranges} = 12 \times 150 = 1800
\]

### Step 2: Calculate the total number of nectarines

- Number of boxes: \( 16 \)
- Nectarines per box: \( 30 \)

\[
\text{Total number of nectarines} = 16 \times 30 = 480
\]

### Step 3: Add the totals from both fruits

\[
\text{Total number of pieces of fruit} = \text{Total number of oranges} + \text{Total number of nectarines}
\]
\[
= 1800 + 480 = 2280
\]

Thus, the total number of pieces of fruit is **2280**.


In [4]:
# Cell 4 - extract generated answer
import re

def extract_answer(text):
    # Try #### format first
    match = re.search(r'#### (\d+)', text)
    if match:
        return match.group(1)
    
    # Try **$195** or **195** (bold markdown format)
    match = re.search(r'\$(\d+)', text)
    if match:
        return match.group(1)
    
    # Fallback: last number in text
    matches = re.findall(r'\d+', text)
    return matches[-1] if matches else None

In [5]:
# Cell 5 - test extract generated answer
extracted_answer = extract_answer(generated_answer)
print(extracted_answer)

2280


In [6]:
# Cell 6- Load GSM-8K dataset
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")
train_set = dataset["train"]
test_set = dataset["test"]

In [7]:
# Cell 7 - Extract correct answer
def extract_correct_answer(text):
    match = re.search(r'#### (\d+)', text)
    return match.group(1) if match else None

In [8]:
# Cell 8 - test extract correct answer
example = train_set[0]
question, answer = example["question"], example["answer"]
correct_answer = extract_correct_answer(answer)
print(correct_answer)

72


In [9]:
# Cell 9 - embed every single qn
from sentence_transformers import SentenceTransformer

t_model = SentenceTransformer("all-MiniLM-L6-v2")

embedded_train_set = t_model.encode([example["question"] for example in train_set], convert_to_tensor=True)
answer_train_set = [extract_correct_answer(example["answer"]) for example in train_set]

print(embedded_train_set)
print(answer_train_set)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[ 0.0161, -0.0294,  0.0585,  ..., -0.0822, -0.0750,  0.0240],
        [ 0.0255,  0.0707,  0.0451,  ..., -0.0687,  0.0046,  0.0247],
        [ 0.0323,  0.1454, -0.0100,  ...,  0.0442,  0.0483, -0.0376],
        ...,
        [ 0.0633,  0.0434, -0.0342,  ...,  0.0359,  0.0286,  0.0095],
        [ 0.0009,  0.0707,  0.0626,  ..., -0.0355,  0.0148,  0.0245],
        [ 0.0283,  0.0047,  0.0159,  ..., -0.0197, -0.0384,  0.0182]],
       device='cuda:0')
['72', '10', '5', '42', '624', '35', '48', '16', '41', '990', '121', '5', '85', '35', '5', '448000', '800', '43', '16', '16', '38', '1080', '7', '5', '62', '110', '400', '400', '8', '1000', '6', '1200', '10', '34', '5250', '36', '15', '5', '9', '15', '476', '500', '99', '60', '300', '99', '1920', '15', '10', '48', '5', '160', '5', '36', '11', '75', '45', '2', '320', '120', '96', '200', '15', '59', '840', '558', '520', '6', '90', '49', '19', '25', '54', '3', '28', '15', '768', '85', '4', '70', '100', '14', '700', '54', '90', '5', '6', '6

In [10]:
# cell 10 - find similar questions
from sentence_transformers import util

def find_similar_questions(input_question: str, top_k: int = 3):
    global questions, gsm8k
    t_model = SentenceTransformer("all-MiniLM-L6-v2")
    query_embedding = t_model.encode(input_question, convert_to_tensor=True)
    hits = util.semantic_search(query_embedding, embedded_train_set, top_k=top_k)[0] # cosine similarity
    return [
        {"score": h["score"], "question": train_set[h["corpus_id"]]["question"], "answer": train_set[h["corpus_id"]]["answer"]}
        for h in hits
    ]

In [11]:
# Cell 11 - test find similar questions
similar = find_similar_questions("There are 12 crates that each contain 150 oranges. \
There are 16 boxes that each hold 30 nectarines. \
How many pieces of fruit are in total?")
extracted_similar = []
for k in similar:
    print(k['score'])
    extracted_similar.append(k['question'])
    extracted_similar.append(extract_correct_answer(k['answer']))
extracted_similar

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.9931805729866028
0.7262271642684937
0.6928146481513977


['There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?',
 '2280',
 'If a basket is capable of holding 40 fruit altogether, and there are 3 times as many apples as oranges, how many oranges are there?',
 '10',
 'There are 10 bags with 30 oranges each on a truck. A total of 50 pieces of oranges are rotten. Thirty pieces of oranges will be kept for making orange juice and the rest will be sold. How many pieces of oranges will be sold?',
 '220']

In [12]:
# Cell 12 - few shot prompting
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"# Larger model for bigger accuracy

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda"
)

def few_shot_prompt(question):
    questions = find_similar_questions(question)
    prompt = f"""Solve the following math problems. Always end your answer with #### <number> where <number> is the final answer.

Q: {questions[0]['question']}
A: {questions[0]['answer']}

Q: {questions[1]['question']}
A: {questions[1]['answer']}

Q: {questions[2]['question']}
A: {questions[2]['answer']}

Q: {question}
A:"""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # ← must not be commented out

    output = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.3,
        top_p=0.95
    )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = output[0][input_length:]  # ← only decode new tokens
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [13]:
# cell 13- test few shot prompting
answer = few_shot_prompt("There are 12 crates that each contain 150 oranges. \
There are 16 boxes that each hold 30 nectarines. \
How many pieces of fruit are in total?")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The final answer is 2280.


In [17]:
# cell 14 -test accuracy with zero shot prompt
count, correct = 0,0 
for i in test_set:
    q, a = i["question"], extract_correct_answer(i["answer"])
    ans  = extract_answer(zero_shot_prompt(q))
    count += 1
    if ans == a:
        correct += 1
print(correct, count, correct/count)

223 1319 0.16906747536012132


In [19]:
# cell 15 -test accuracy with few shot prompt
count, correct = 0,0 
for i in test_set:
    q, a = i["question"], extract_correct_answer(i["answer"])
    ans  = extract_answer(few_shot_prompt(q))
    # print(ans, a)
    count += 1
    if ans == a:
        correct += 1
print(correct, count, correct/count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


145 1319 0.10993176648976498


In [20]:
print(correct, count, correct/count)

145 1319 0.10993176648976498
